## Imports

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

import torch
from torch_geometric.data import Data
from scipy.spatial import Delaunay

import networkx as nx
from torch_geometric.utils import to_networkx

## GRAPHS FROM SYNTHETIC DATA

In [ ]:
synthetic_1 = np.load('../data/preprocessing/normalized/normalized_local_close.npy')[:15000]
synthetic_2 = np.load('../data/preprocessing/normalized/normalized_local_far.npy')[:15000]
original = np.load('../data/preprocessing/normalized/normalized_local_original.npy')[:15000]

In [ ]:
def point_to_polyline_distance_vectorized(points, polyline):

    A = polyline[:-1]        # (M-1, 2)
    B = polyline[1:]         # (M-1, 2)
    AB = B - A               # (M-1, 2)
    AB_norm_sq = np.sum(AB ** 2, axis=1)  # (M-1,)

    # Expand dims for broadcasting
    P = points[:, None, :]   # (N,1,2)
    A = A[None, :, :]        # (1,M-1,2)
    AB = AB[None, :, :]      # (1,M-1,2)

    AP = P - A                      # (N,M-1,2)
    t = np.sum(AP * AB, axis=2) / (AB_norm_sq + 1e-12)  # (N, M-1)
    t = np.clip(t, 0.0, 1.0)

    proj = A + t[..., None] * AB    # (N,M-1,2)
    dist = np.linalg.norm(P - proj, axis=2)  # (N, M-1)

    return np.min(dist, axis=1)  # (N,)

In [ ]:
def compute_orientation_angles(coords):
    
    dx = np.diff(coords[:, 0], append=coords[-1, 0])
    dy = np.diff(coords[:, 1], append=coords[-1, 1])
    angles = np.arctan2(dy, dx)
    angles = (angles + np.pi) / (2 * np.pi)  # normalize 0–1
    return angles.reshape(-1, 1)

In [ ]:
def make_graphs(original, synthetic_1, synthetic_2):
    
    num_seqs = original.shape[0]
    graphs = []

    for seq_idx in range(num_seqs):
        seq_len = original.shape[1]   # auto-detect length
         
        # Extract polylines
        orig = original[seq_idx][:seq_len]         # (N,2)
        syn1 = synthetic_1[seq_idx][:seq_len]      # (N,2)
        syn2 = synthetic_2[seq_idx][:seq_len]      # (N,2)
         
        # Compute node features
        feats = []

        # ----- Original line features -----
        angles_o = compute_orientation_angles(orig)
        dist_o = point_to_polyline_distance_vectorized(orig, syn1).reshape(-1, 1)

        feats_o = np.hstack([
            np.zeros((seq_len, 1)),      # line_id = 0
            orig,                        # x,y
            angles_o,                    # orientation
            dist_o                       # dist to synthetic_1
        ])
        feats.append(feats_o)

        # ----- Synthetic_1 line features -----
        angles_s1 = compute_orientation_angles(syn1)
        dist_s1 = point_to_polyline_distance_vectorized(syn1, orig).reshape(-1, 1)

        feats_s1 = np.hstack([
            np.ones((seq_len, 1)),       # line_id = 1
            syn1,
            angles_s1,
            dist_s1
        ])
        feats.append(feats_s1)

        # Stack node features
        x = np.vstack(feats)  # (2*N, feature_dim)
         
        # Build edges
        edge_src = []
        edge_dst = []

        # Within original
        idx_offset_o = 0
        for i in range(seq_len - 1):
            edge_src += [idx_offset_o + i, idx_offset_o + i + 1]
            edge_dst += [idx_offset_o + i + 1, idx_offset_o + i]

        # Within synthetic_1
        idx_offset_s1 = seq_len
        for i in range(seq_len - 1):
            edge_src += [idx_offset_s1 + i, idx_offset_s1 + i + 1]
            edge_dst += [idx_offset_s1 + i + 1, idx_offset_s1 + i]

        # Cross edges original[i] <-> synthetic_1[i]
        for i in range(seq_len):
            edge_src += [i,             idx_offset_s1 + i]
            edge_dst += [idx_offset_s1 + i, i]

        edge_index = np.vstack([edge_src, edge_dst]).astype(np.int64)
         
        # Target: shift of synthetic_1 → synthetic_2
        shift = syn2 - syn1         # (N,2)

        y = np.zeros((2 * seq_len, 2), dtype=np.float32)
        y[idx_offset_s1:idx_offset_s1 + seq_len] = shift  # only syn1 nodes

         
        # Convert to PyTorch
        data = Data(
            x=torch.tensor(x, dtype=torch.float),
            edge_index=torch.tensor(edge_index, dtype=torch.long),
            y=torch.tensor(y, dtype=torch.float)
        )
        graphs.append(data)

    return graphs


In [ ]:
graphs_seq = make_graphs(original, synthetic_1, synthetic_2)

In [ ]:
def make_graphs_delaunay(original, synthetic_1, synthetic_2):

    num_seqs = original.shape[0]
    graphs = []

    for seq_idx in range(num_seqs):
        seq_len = original.shape[1]

        # Extract polylines
        orig = original[seq_idx][:seq_len]    # (N,2)
        syn1 = synthetic_1[seq_idx][:seq_len]
        syn2 = synthetic_2[seq_idx][:seq_len]
         
        # Compute node features
        feats = []

        # ---- Original line ----
        angles_o = compute_orientation_angles(orig)
        dist_o = point_to_polyline_distance_vectorized(orig, syn1).reshape(-1, 1)

        feats_o = np.hstack([
            np.zeros((seq_len, 1)),  # line_id = 0
            orig,
            angles_o,
            dist_o
        ])
        feats.append(feats_o)

        # ---- Synthetic line 1 ----
        angles_s1 = compute_orientation_angles(syn1)
        dist_s1 = point_to_polyline_distance_vectorized(syn1, orig).reshape(-1, 1)

        feats_s1 = np.hstack([
            np.ones((seq_len, 1)),  # line_id = 1
            syn1,
            angles_s1,
            dist_s1
        ])
        feats.append(feats_s1)

        # Combine features
        x = np.vstack(feats)  # (2N, F)
         
        # Build edges – sequential edges + Delaunay triangulation
        edge_src = []
        edge_dst = []

        # Sequential edges in original line
        idx_offset_o = 0
        for i in range(seq_len - 1):
            edge_src += [idx_offset_o + i, idx_offset_o + i + 1]
            edge_dst += [idx_offset_o + i + 1, idx_offset_o + i]

        # Sequential edges in synthetic_1 line
        idx_offset_s1 = seq_len
        for i in range(seq_len - 1):
            edge_src += [idx_offset_s1 + i, idx_offset_s1 + i + 1]
            edge_dst += [idx_offset_s1 + i + 1, idx_offset_s1 + i]
         
        # Delaunay triangulation edges spanning BOTH lines
        all_points = np.vstack([orig, syn1])  # shape: (2N,2)
        tri = Delaunay(all_points)
        triangles = tri.simplices  # (T,3)

        tri_edges = set()
        for a, b, c in triangles:
            tri_edges.add(tuple(sorted((a, b))))
            tri_edges.add(tuple(sorted((b, c))))
            tri_edges.add(tuple(sorted((c, a))))

        for u, v in tri_edges:
            edge_src += [u, v]
            edge_dst += [v, u]

        # Final edge index
        edge_index = np.vstack([edge_src, edge_dst]).astype(np.int64)

        # Targets (shift syn1 → syn2)
        shift = syn2 - syn1  # (N,2)

        y = np.zeros((2 * seq_len, 2), dtype=np.float32)
        y[idx_offset_s1:idx_offset_s1 + seq_len] = shift  # only synthetic_1 nodes
         
        # Create PyG Data
        data = Data(
            x=torch.tensor(x, dtype=torch.float),
            edge_index=torch.tensor(edge_index, dtype=torch.long),
            y=torch.tensor(y, dtype=torch.float)
        )

        graphs.append(data)

    return graphs

In [ ]:
graphs_delaunay = make_graphs_delaunay(original, synthetic_1, synthetic_2)

### Save Results

In [ ]:
#torch.save(graphs_seq, f'../data/final_dataset/graph/graphs_sequential_new.pt')
#torch.save(graphs_delaunay, f'../data/final_dataset/graph//graphs_delaunay_new.pt')

### Ploting Graphs

In [ ]:
def plot_graph(data):
    # Convert PyG Data -> NetworkX
    G = to_networkx(data, to_undirected=True)

    # Extract node positions (x,y from features)
    pos = {i: (float(data.x[i][1]), float(data.x[i][2])) for i in range(data.num_nodes)}

    # Node colors by line_id
    color_map = ["#143642", "#EC9A29", "#A8201A"]
    colors = [color_map[int(data.x[i][0].item())] for i in range(data.num_nodes)]

    plt.figure(figsize=(8, 6))
    plt.title(f'Constructed Graph')
    plt.xlabel('X')
    plt.ylabel('Y')
    nx.draw(G, pos,
            node_size=10,
            node_color=colors,
            edge_color="#B3B5B6A6",
            alpha=0.8)
    plt.show()

In [ ]:
graph = graph_d

for i in range(10,20):
    plot_graph(graph[i])